# Python for Energy System Modelling

**Guest workshop, MSc Sustainable Engineering: Energy, Nottingham Trent University**
Ammar Ahmed Wafad

This notebook accompanies the workshop slides. You will use real solar data for Nottingham to size a home battery, then use the open-source framework PyPSA to see how a carbon price changes which power stations run and what electricity costs.

- **Duration:** about 40 minutes of hands-on work.
- **Prior knowledge:** none in Python; basic energy concepts (kW, kWh, marginal cost).
- **Working style:** groups of three or four. The Modeller runs the notebook, the Reviewer reads each step aloud and checks the method, and the Reporter verifies the numbers and records the group's results. In a group of four, the Second reviewer checks the hints, figures and resources.
- **Outputs:** every figure is saved as a PNG and every table as a CSV in the `outputs/` folder.

> **Instructor version.** The task blanks are already completed in this copy. Share the student version with the class.

## How to use this notebook

**Where to run it.** On an NTU computer, open **Anaconda Navigator** from the NTU Software Hub, launch **JupyterLab**, and open this notebook from a folder you can save to, such as Documents or your H: drive. On your own laptop you can also open it in **Google Colab** (colab.research.google.com, then File > Upload notebook). The notebook works the same way in both.

1. **Run cells in order, from top to bottom.** Click a grey code cell and press **Shift + Enter**. A number appears in the brackets on the left when the cell has finished.
2. **Read the text cell above each code cell first.** It explains what the code does and what to look for in the output.
3. **Cells marked `TASK` need you to type something.** Everything else is ready to run.
4. **If something goes wrong,** read the last line of the error message first. Most errors are a typo in a name or a missing bracket. The menu commands below help you recover.
5. **Your results.** Every figure and table is saved in the `outputs/` folder next to this notebook as you go. The last cell in Section 6 collects everything in one zip file.

| To do this | JupyterLab (Anaconda) | Google Colab |
|---|---|---|
| Run the selected cell | Shift + Enter | Shift + Enter |
| Run every cell above the selected one | Run > Run All Above Selected Cell | Runtime > Run before |
| Run the whole notebook | Run > Run All Cells | Runtime > Run all |
| Start again from a clean state | Kernel > Restart Kernel and Run All Cells | Runtime > Restart session and run all |
| See the saved files | File Browser (folder icon, top left) | Files (folder icon, left) |

### Learning outcomes

By the end of the session you should be able to:

1. Load an hourly energy time series into a pandas DataFrame and check it.
2. Write a simple time-stepping model of a home battery and calculate the self-consumption rate, self-sufficiency rate, grid imports and net energy cost.
3. Explain how battery losses and power limits change the result, and why an idealised model gives an upper bound.
4. Build and solve a small least-cost (economic) dispatch model in PyPSA.
5. Explain how a carbon price changes the merit order, the clearing price and emissions, and state the main limitations of each model.

### Contents

1. Setup
2. Python in five minutes
3. Data: solar generation and household demand
4. Task 1: a home with solar and a battery
5. Task 2: the merit order and a carbon price
6. Summary, limitations and downloads
7. References

## 1. Setup

The first cell installs PyPSA, the open-source power system modelling framework used in Task 2. It installs PyPSA only if it is missing, which takes one or two minutes the first time. **Run it now** and keep reading while it works.

The cell does not change any library that is already installed. It reads the version of pandas that Anaconda or Colab provides and installs the tested PyPSA release that matches it (PyPSA 1.2.1 for pandas 2, PyPSA 1.4.0 for pandas 3). If you see a note saying you may need to restart the kernel, you can ignore it.

In [ ]:
import importlib
import importlib.util
import site
import sys
from importlib.metadata import version

IN_COLAB = "google.colab" in sys.modules

if sys.version_info < (3, 11):
    raise RuntimeError(f"This notebook needs Python 3.11 or newer, but this kernel runs Python "
                       f"{sys.version.split()[0]}. Choose a newer kernel (Kernel > Change Kernel) and run this cell again.")

if importlib.util.find_spec("pypsa") is None:
    # Choose the tested PyPSA release that matches the installed pandas, and pin pandas
    # and NumPy to their current versions so that pip never changes them.
    pandas_version, numpy_version = version("pandas"), version("numpy")
    pypsa_pin = "pypsa==1.4.0" if int(pandas_version.split(".")[0]) >= 3 else "pypsa==1.2.1"
    pandas_pin, numpy_pin = f"pandas=={pandas_version}", f"numpy=={numpy_version}"
    print(f"Installing {pypsa_pin} for pandas {pandas_version}. This takes one or two minutes ...")
    %pip install -q --no-warn-script-location {pypsa_pin} {pandas_pin} {numpy_pin}

    # On a managed computer pip may install into your user folder; make sure Python can see it.
    # Put that folder first, so any newer library installed there is the one Python uses.
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.insert(0, user_site)
        site.addsitedir(user_site)
    importlib.invalidate_caches()

if importlib.util.find_spec("pypsa") is None:
    raise ImportError("PyPSA is not available yet. Use Kernel > Restart Kernel (JupyterLab) or "
                      "Runtime > Restart session (Colab), then run this cell again. "
                      "If it still fails, check the internet connection or ask the instructor.")
print(f"PyPSA {version('pypsa')} is ready.")

### 1.1 Libraries, settings and helper functions

This cell imports the libraries, sets a consistent style for the figures and defines three small helper functions:

- `save_figure()` saves a figure as a high-resolution PNG in `outputs/figures`.
- `save_table()` saves a table as a CSV file in `outputs/tables`.
- `show_table()` displays a table neatly in the notebook.

You do not need to change anything here.

In [ ]:
import json
import logging
import shutil
import sys
import warnings
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import display

# Keep the solver output short so the results are easy to read.
logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = Path("data")
FIG_DIR = Path("outputs/figures")
TABLE_DIR = Path("outputs/tables")
for folder in (DATA_DIR, FIG_DIR, TABLE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# One colour per technology, used in every figure.
COLOURS = {
    "demand": "#1F2933",
    "solar": "#E8A202",
    "battery": "#6E9F1A",
    "grid": "#2F86C9",
    "export": "#9AA5B1",
    "wind": "#2BB3C0",
    "nuclear": "#A26CB8",
    "coal": "#6B7480",
    "gas": "#D9483B",
    "price": "#1F2933",
}

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#E4E7EB",
    "grid.linewidth": 0.8,
    "legend.frameon": False,
})
pd.set_option("display.precision", 2)

# Record software versions, so results can be reproduced later.
import importlib.metadata as metadata
import pypsa

print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"matplotlib {metadata.version('matplotlib')} | PyPSA {pypsa.__version__} | HiGHS (highspy) {metadata.version('highspy')}")


def save_figure(fig, name):
    # Save a figure as a 300 dpi PNG in outputs/figures.
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print(f"Figure saved: {path}")


def save_table(df, name, index=True):
    # Save a table as CSV in outputs/tables.
    path = TABLE_DIR / f"{name}.csv"
    df.to_csv(path, index=index)
    print(f"Table saved: {path}")


def show_table(df, decimals=2):
    # Display a table rounded for reading. The saved CSV keeps full precision.
    display(df.round(decimals))


print("Setup complete.")

## 2. Python in five minutes

These are the only five pieces of Python you need for Task 1. Run each cell and compare the output with the code.

**1. Variables** store values under a name. Text after `#` is a comment, which Python ignores.

In [ ]:
pv_kwh = 1.8        # solar generation in one hour, kWh
demand_kwh = 0.6    # household demand in the same hour, kWh

net_kwh = pv_kwh - demand_kwh
print("Surplus this hour:", round(net_kwh, 2), "kWh")

**2. A DataFrame** is a table, like a spreadsheet. Each column has a name and each row has an index. This is the main data structure in pandas.

In [ ]:
example = pd.DataFrame({
    "hour": ["06:00", "12:00", "18:00", "21:00"],
    "pv_kwh": [0.1, 2.4, 0.3, 0.0],
    "demand_kwh": [0.4, 0.5, 1.1, 0.7],
})
example

**3. A `for` loop** repeats the same steps for every item. Here it visits each row of the table in turn, which is exactly how the battery model in Task 1 steps through the 8,760 hours of a year.

**4. An `if` statement** makes a decision. Here it checks whether the home has a surplus or a shortfall in each hour.

In [ ]:
total_import = 0.0

for t in range(len(example)):
    net = example["pv_kwh"][t] - example["demand_kwh"][t]
    if net < 0:
        total_import += -net   # a shortfall is bought from the grid

print("Energy bought from the grid:", total_import, "kWh")

**5. `print` and `round`** report a result. Always print the units with the number.

**Reading errors.** Errors are normal, and reading them is a skill. A Python error message can be long, but the useful part is the **last line**. The next cell asks for a column that does not exist and prints the error instead of stopping. `KeyError: 'demand'` means there is no column called `demand`. The real name is `demand_kwh`, which the second line uses correctly.

In [ ]:
try:
    peak_demand = example["demand"].max()          # wrong column name
except KeyError as error:
    print("KeyError:", error)

peak_demand = example["demand_kwh"].max()          # correct column name
print("Peak demand:", round(peak_demand, 2), "kWh")

## 3. Data: solar generation and household demand

A model is only as good as its inputs. This section builds the hourly data set used in Task 1 and saves it as `data/home.csv`.

| Input | Source | Notes |
|---|---|---|
| Solar generation | PVGIS 5.3, European Commission Joint Research Centre | Modelled hourly AC output of a 4 kWp roof system at NTU Clifton campus, from satellite-derived irradiance (PVGIS-SARAH3) for the single weather year 2019 |
| Household demand | Synthetic profile built in this notebook | A typical UK daily and seasonal shape in UK local time, scaled to Ofgem's 2026 medium Typical Domestic Consumption Value (TDCV) for single-rate homes (Profile Class 1): 2,500 kWh per year |

### 3.1 Assumptions

All the main assumptions sit in one place, so they are easy to find and change. Each one states its unit and where it comes from.

- **kWp** (kilowatt-peak) is the DC power the panels produce under Standard Test Conditions: 1,000 W/m² of sunlight on the panel, a cell temperature of 25 °C and the AM1.5 reference spectrum. Real output is almost always lower.
- **Azimuth** in PVGIS is measured from due south, so 0° means south-facing. Many other tools, including pvlib, measure from north, so south is 180° there. Mixing the two conventions turns a south-facing roof into a north-facing one.
- **Tariffs** are typical published values: the Ofgem price cap unit rate for October to December 2026 and a Smart Export Guarantee (SEG) rate. Open SEG tariffs ranged from about 3 to 13 p/kWh in 2026, so the export price is a sensible central value, not a single correct number.

In [ ]:
# Location: Nottingham Trent University, Clifton campus (approximate)
LATITUDE = 52.912
LONGITUDE = -1.184

# Rooftop solar system (inputs to PVGIS)
PV_PEAK_KW = 4.0          # peak power at Standard Test Conditions, kWp
PV_SYSTEM_LOSS = 14       # PVGIS default overall system loss, % (cables, inverter, soiling, ageing)
PV_TILT = 35              # slope: module angle from the horizontal, degrees
PV_AZIMUTH = 0            # PVGIS convention: 0 = south, -90 = east, 90 = west (not a compass bearing)
WEATHER_YEAR = 2019       # one actual weather year, not a typical year; 8,760 hours

# Household demand
ANNUAL_DEMAND_KWH = 2500  # Ofgem medium TDCV, Profile Class 1, from 1 July 2026
DEMAND_TIMEZONE = "Europe/London"  # household routines follow the local clock (GMT and BST)

# Home battery
BATTERY_KWH = 5.0             # usable capacity, kWh
BATTERY_POWER_KW = 3.0        # maximum charge and discharge power, kW (realistic case)
ROUND_TRIP_EFFICIENCY = 0.90  # AC to AC, energy out divided by energy in (realistic case)

# Tariffs, as at October 2026. Prices change, so check them before reusing.
IMPORT_PRICE = 0.2632     # GBP/kWh, Ofgem price cap unit rate, 1 Oct to 31 Dec 2026 (26.32 p/kWh)
STANDING_CHARGE = 0.5483  # GBP/day, Ofgem price cap, 1 Oct to 31 Dec 2026; unchanged by solar
EXPORT_PRICE = 0.12       # GBP/kWh, Smart Export Guarantee tariff (Outgoing Octopus fixed rate from 1 Mar 2026)

### 3.2 Solar generation from PVGIS

PVGIS (Photovoltaic Geographical Information System) is a free service from the European Commission's Joint Research Centre. Its hourly service combines satellite irradiance data with a PV performance model and returns the output of a system you describe.

The function below:

1. checks whether the data has already been downloaded to `data/` and, if so, uses that copy;
2. otherwise requests the data from the PVGIS API;
3. converts the result to an hourly pandas Series in kWh.

A few details matter when you use this data:

- **It is modelled, not measured.** The irradiance comes from images taken by Meteosat satellites (the SARAH-3 data record from EUMETSAT's CM SAF), which PVGIS converts to PV output with a standard performance model.
- **It is AC output after losses.** PVGIS has already applied the 14 % system loss, including the inverter, so no further inverter loss should be applied.
- **Time and units.** Timestamps are in UTC. Each value `P` is the power in watts at ten minutes past the hour, taken as the average for that hour. An average of 1 kW for one hour delivers 1 kWh, so dividing by 1,000 gives kWh for the hour.

In [ ]:
PVGIS_URL = "https://re.jrc.ec.europa.eu/api/v5_3/seriescalc"
PVGIS_CACHE = DATA_DIR / f"pvgis_hourly_{WEATHER_YEAR}.json"

# Backup copy of the finished data set. Used only if PVGIS cannot be reached.
BACKUP_CSV_URL = ""


def download_pvgis(cache_path=PVGIS_CACHE):
    # Return the PVGIS hourly response, downloading it once and caching it.
    if cache_path.exists():
        print(f"Using the saved PVGIS file: {cache_path}")
        return json.loads(cache_path.read_text(encoding="utf-8"))

    params = {
        "lat": LATITUDE, "lon": LONGITUDE,
        "peakpower": PV_PEAK_KW, "loss": PV_SYSTEM_LOSS,
        "angle": PV_TILT, "aspect": PV_AZIMUTH,
        "startyear": WEATHER_YEAR, "endyear": WEATHER_YEAR,
        "pvcalculation": 1, "outputformat": "json",
    }
    print("Downloading hourly solar data from PVGIS ...")
    response = requests.get(PVGIS_URL, params=params, timeout=60)
    response.raise_for_status()
    pvgis_json = response.json()            # check the reply is valid PVGIS data before saving it
    if "outputs" not in pvgis_json:
        raise ValueError("PVGIS replied, but without hourly data.")
    cache_path.write_text(response.text, encoding="utf-8")
    print(f"Saved to {cache_path}")
    return pvgis_json


def parse_pvgis(pvgis_json):
    # Convert the PVGIS response into an hourly Series of PV generation in kWh.
    hourly = pd.DataFrame(pvgis_json["outputs"]["hourly"])
    timestamps = pd.to_datetime(hourly["time"], format="%Y%m%d:%H%M").dt.floor("h")
    pv_kwh = pd.Series(hourly["P"].to_numpy() / 1000, index=timestamps, name="pv_kwh")
    pv_kwh.index.name = "timestamp_utc"
    return pv_kwh


try:
    pvgis = download_pvgis()
    pv = parse_pvgis(pvgis)
    meteo = pvgis["inputs"].get("meteo_data", {})
    print(f"Radiation database: {meteo.get('radiation_db', 'not stated')}")
except Exception as error:
    if not BACKUP_CSV_URL:
        raise RuntimeError(
            "PVGIS could not be reached and no backup file is set. "
            "Check the internet connection and run this cell again."
        ) from error
    print(f"PVGIS unavailable ({error}). Loading the backup data set instead.")
    backup = pd.read_csv(BACKUP_CSV_URL, index_col="timestamp_utc", parse_dates=True)
    backup.to_csv(DATA_DIR / "home.csv")
    pv = backup["pv_kwh"]

specific_yield = pv.sum() / PV_PEAK_KW                  # kWh/kWp
capacity_factor = pv.sum() / (PV_PEAK_KW * len(pv))     # AC energy relative to DC peak power

print(f"Hours of data: {len(pv):,}")
print(f"Annual generation: {pv.sum():,.0f} kWh")
print(f"Specific yield: {specific_yield:,.0f} kWh/kWp (modelled)")
print(f"Capacity factor: {100 * capacity_factor:.1f} %")

**Check the result.** The **specific yield** is the annual energy per kWp installed. For an unshaded, south-facing roof in England, modelled values of roughly 850 to 1,000 kWh/kWp are typical, depending on the weather year. Measured systems usually deliver less, because real roofs have shading, dirt, outages and less ideal orientations: across UK Feed-in Tariff installations the median load factor in 2024/25 was 9.2 %, about 800 kWh/kWp (DESNZ, 2025). If your number is far outside these ranges, check the inputs and the units.

The **capacity factor** expresses the same result as a share of the energy the system would produce at full peak power all year.

### 3.3 Household demand

There is no free, ready-made hourly demand file for a single UK home, so we build a transparent synthetic profile. It is not a measurement of a real house. It reproduces the features that matter for this task:

- a low overnight base load (fridge, freezer and standby devices);
- a morning peak and a larger evening peak, when most people are at home;
- higher demand in winter, mainly from lighting and more time indoors;
- small random hour-to-hour variation, with a fixed random seed so that everyone gets the same data.

The profile is then scaled so that the annual total equals `ANNUAL_DEMAND_KWH`. Ofgem's **Typical Domestic Consumption Value** is based on the median household, not the average, and Ofgem describes it as an indicative estimate rather than a precise measure.

**Time zone.** People live by the local clock, so the daily routine is built in UK local time, which moves between GMT and British Summer Time (BST), and then converted to UTC to line up with the solar data.

**An important limitation.** A smooth, averaged profile like this has no short spikes from kettles, showers or ovens. Real homes have them, and they make it harder to use solar directly. Smooth profiles and hourly time steps therefore tend to **overstate** self-consumption. The seasonal shape here is an assumption and has not been checked against Elexon's Profile Class 1 data.

In [ ]:
# Relative demand for each hour of the day (00:00 to 23:00).
WEEKDAY_SHAPE = np.array([
    0.45, 0.40, 0.38, 0.37, 0.37, 0.40, 0.60, 0.90, 0.85, 0.65, 0.60, 0.62,
    0.66, 0.62, 0.60, 0.66, 0.85, 1.15, 1.35, 1.30, 1.15, 0.95, 0.75, 0.55,
])
WEEKEND_SHAPE = np.array([
    0.50, 0.42, 0.39, 0.38, 0.38, 0.40, 0.45, 0.60, 0.80, 0.90, 0.90, 0.92,
    0.95, 0.88, 0.82, 0.85, 0.95, 1.15, 1.30, 1.25, 1.10, 0.95, 0.80, 0.60,
])


def make_household_demand(index_utc, annual_kwh, seed=42):
    # Build an hourly demand profile in kWh on an hourly UTC index.
    # The daily routine follows UK local time (GMT in winter, BST in summer).
    local = index_utc.tz_localize("UTC").tz_convert(DEMAND_TIMEZONE)
    hour = local.hour.to_numpy()
    weekend = local.dayofweek.to_numpy() >= 5
    day_of_year = local.dayofyear.to_numpy()

    shape = np.where(weekend, WEEKEND_SHAPE[hour], WEEKDAY_SHAPE[hour])

    # Seasonal factor: about +20 % in mid-winter and -20 % in mid-summer.
    seasonal = 1 + 0.20 * np.cos(2 * np.pi * (day_of_year - 15) / 365)

    # Lighting extends the evening peak in the darker months.
    winter_evening = (hour >= 16) & (hour <= 19) & (seasonal > 1.1)
    shape = np.where(winter_evening, shape * 1.15, shape)

    rng = np.random.default_rng(seed)
    noise = rng.lognormal(mean=0.0, sigma=0.12, size=len(index_utc))

    profile = shape * seasonal * noise
    demand = profile / profile.sum() * annual_kwh
    return pd.Series(demand, index=index_utc, name="demand_kwh")


demand = make_household_demand(pv.index, ANNUAL_DEMAND_KWH)
monthly_demand = demand.resample("MS").sum()
print(f"Annual demand: {demand.sum():,.0f} kWh")
print(f"Overnight base load (lowest hour): {demand.min() * 1000:,.0f} W")
print(f"Highest demand in any hour: {demand.max():.2f} kWh")
print(f"January to July demand ratio: {monthly_demand.iloc[0] / monthly_demand.iloc[6]:.2f}")

### 3.4 Combine, check and save

We place both series in one DataFrame, check it for problems and save it as `data/home.csv`. This is the file Task 1 reads.

Checking data before using it is a habit worth building early: count the rows, look for missing values and make sure nothing is negative.

In [ ]:
home = pd.concat([pv, demand], axis=1)

checks = {
    "8,760 hourly rows": len(home) == 8760,
    "No missing values": not home.isna().any().any(),
    "No negative values": bool((home >= 0).all().all()),
    "No duplicate hours": not home.index.duplicated().any(),
}
for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {check}")
assert all(checks.values()), "Fix the data problem above before continuing."

home.to_csv(DATA_DIR / "home.csv")
print(f"\nSaved {DATA_DIR / 'home.csv'}")
home.head()

### 3.5 Look at the data

Two views tell you most of what you need to know about this home.

**Figure 1** compares a summer week with a winter week. Look at when solar peaks and when demand peaks. The gap between them is the opportunity for a battery.

In [ ]:
weeks = {"Summer week (17 to 23 June)": ("2019-06-17", "2019-06-23"),
         "Winter week (9 to 15 December)": ("2019-12-09", "2019-12-15")}

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharey=True)
for ax, (title, (start, end)) in zip(axes, weeks.items()):
    week = home.loc[start:f"{end} 23:00"]
    ax.fill_between(week.index, week["pv_kwh"], color=COLOURS["solar"], alpha=0.35, linewidth=0)
    ax.plot(week.index, week["pv_kwh"], color=COLOURS["solar"], linewidth=1.2, label="Solar generation")
    ax.plot(week.index, week["demand_kwh"], color=COLOURS["demand"], linewidth=1.2, label="Demand")
    ax.set_title(title, loc="left")
    ax.set_ylabel("Energy per hour (kWh)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))
axes[0].legend(loc="upper right", ncol=2)
fig.suptitle("Figure 1. Hourly solar generation and household demand", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save_figure(fig, "fig01_summer_winter_weeks")
plt.show()

**Figure 2** shows the monthly totals. Over a year the panels generate more than the home uses, but most of it arrives in summer and around midday. Annual totals alone can be misleading, which is why we model every hour.

In [ ]:
monthly = home.resample("MS").sum()
monthly.index = monthly.index.strftime("%b")

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(monthly))
ax.bar(x - 0.2, monthly["pv_kwh"], width=0.4, color=COLOURS["solar"], label="Solar generation")
ax.bar(x + 0.2, monthly["demand_kwh"], width=0.4, color=COLOURS["demand"], label="Demand")
ax.set_xticks(x, monthly.index)
ax.set_ylabel("Energy (kWh per month)")
ax.set_title("Figure 2. Monthly solar generation and household demand", loc="left")
ax.legend(loc="upper right", ncol=2)
ax.grid(axis="x", visible=False)
fig.tight_layout()
save_figure(fig, "fig02_monthly_energy")
plt.show()

monthly_table = monthly.rename(columns={"pv_kwh": "Solar generation (kWh)", "demand_kwh": "Demand (kWh)"})
show_table(monthly_table, 0)
save_table(monthly_table, "tab01_monthly_energy")

## 4. Task 1: a home with solar and a battery

**Question.** How much does a 5 kWh battery raise the self-sufficiency rate of a Nottingham home with 4 kWp of solar?

**Time:** 15 minutes. **Roles:** the Modeller runs the notebook, the Reviewer reads each step aloud and checks the method, the Reporter verifies every number and unit. In a group of four, the Second reviewer checks the hints, figures and resources.

**Results form.** The Reporter records your group's results in the Google Form (QR code on the slide): ten short questions across both tasks. The two **prediction** questions must be answered **before** you run the matching cell.

<details><summary><b>If something goes wrong</b> (common errors and fixes)</summary>

| You see | What to do |
|---|---|
| `NameError: name '____' is not defined` | A blank is still empty. Fill it, run that cell, then run the test again. |
| The test still fails after you fixed a blank | Run the cell that defines the function again. The test uses the version that was last run. |
| `NameError` for `home`, `with_battery` or `MY_GROUP` | An earlier cell has not run. Select this cell, then use **Run > Run All Above Selected Cell** (JupyterLab) or **Runtime > Run before** (Colab). |
| `AssertionError: ... check TASK 1` (or TASK 2) | Your rule lets the battery go above full or below empty. Open the hints in section 4.3. |
| Time to move on, but the battery blanks still give an error | Click in the first code cell of section 4.6 and use **Run > Run Selected Cell and All Below** (JupyterLab), or press Shift + Enter there and then use **Runtime > Run after** (Colab). Everything from section 4.6 onwards works without the blanks. |
| `ValueError: MY_GROUP must be a whole number` | Type the number without quotes, for example `MY_GROUP = 3`. |
| Everything is suddenly undefined | The notebook restarted. Use **Run > Run All Cells** (JupyterLab) or **Runtime > Run all** (Colab), and wait for section 3 to finish. |
| `PVGIS could not be reached` | Wait a minute and run the cell again, or ask the instructor. |

Read the **last line** of an error first: it names the problem.
</details>

### 4.1 Measures of performance

For every hour $t$, the home has solar generation $PV_t$, demand $D_t$, grid import $\text{Import}_t$ and grid export $\text{Export}_t$, all in kWh. The solar energy the home actually uses, directly or after storing it in the battery, is

$$E_\text{self} = \sum_t \left( D_t - \text{Import}_t \right)$$

The two standard measures (Luthander et al., 2015) divide this same quantity by different totals:

$$\text{Self-consumption rate} = \frac{E_\text{self}}{\sum_t PV_t} \qquad \text{the share of the solar generation used by the home}$$

$$\text{Self-sufficiency rate} = \frac{E_\text{self}}{\sum_t D_t} \qquad \text{the share of the demand met by the home's own solar}$$

The self-sufficiency rate is also called the **degree of autarky**. Energy lost inside a battery is counted as solar that was **not** self-consumed, so a lossy battery lowers both measures.

The money side uses two figures:

$$\text{Net energy cost} = \text{import price} \times \sum_t \text{Import}_t \; - \; \text{export price} \times \sum_t \text{Export}_t$$

$$\text{Annual bill} = \text{net energy cost} + 365 \times \text{standing charge}$$

The standing charge is the same with or without solar and a battery, so it never changes the saving. It is shown so the bill looks like a real one. A negative net energy cost means export income is larger than the cost of imports.

A battery cannot create energy. It moves surplus solar from the hours when it is generated to the hours when it is needed, and loses some on the way.

### 4.2 Step 1: the baseline without a battery

Without a battery, solar either serves demand straight away or is exported. Run the cell and note the results. This is your baseline.

In [ ]:
def summarise(name, pv_kwh, demand_kwh, grid_import, grid_export,
              import_price=None, export_price=None):
    # Return the performance measures for one scenario (Luthander et al., 2015).
    self_consumed = demand_kwh.sum() - grid_import.sum()      # solar used directly or via the battery, kWh
    import_price = IMPORT_PRICE if import_price is None else import_price
    export_price = EXPORT_PRICE if export_price is None else export_price
    net_energy_cost = import_price * grid_import.sum() - export_price * grid_export.sum()
    return pd.Series({
        "Self-consumption rate (%)": 100 * self_consumed / pv_kwh.sum(),
        "Self-sufficiency rate (%)": 100 * self_consumed / demand_kwh.sum(),
        "Grid import (kWh)": grid_import.sum(),
        "Grid export (kWh)": grid_export.sum(),
        "Net energy cost (GBP/yr)": net_energy_cost,
        "Annual bill incl. standing charge (GBP/yr)": net_energy_cost + 365 * STANDING_CHARGE,
    }, name=name)


home = pd.read_csv(DATA_DIR / "home.csv", index_col="timestamp_utc", parse_dates=True)

net = home["pv_kwh"] - home["demand_kwh"]
no_battery = pd.DataFrame({
    "pv_direct": np.minimum(home["pv_kwh"], home["demand_kwh"]),
    "battery_discharge": 0.0,
    "grid_import": (-net).clip(lower=0),
    "grid_export": net.clip(lower=0),
    "soc": 0.0,
}, index=home.index)

baseline = summarise("No battery", home["pv_kwh"], home["demand_kwh"],
                     no_battery["grid_import"], no_battery["grid_export"])
show_table(baseline.to_frame(), 1)

### 4.3 Step 2: write the battery rule

This first version is an **idealised** battery: it has no losses and can charge or discharge any amount within an hour. It gives an upper bound on what a battery can do, and section 4.6 then adds the losses and limits of a real product.

The function below steps through the year one hour at a time. In each hour:

1. solar first serves demand directly;
2. if there is **spare solar** (a surplus), it charges the battery, and anything that does not fit is exported;
3. if there is **not enough solar** (a deficit), the battery discharges, and anything it cannot cover is imported from the grid.

`soc` is the battery's **state of charge**, the energy stored in kWh. It must always stay between 0 and `capacity_kwh`.

**TASK.** Replace each `____` with one line of Python. The built-in function `min(a, b)` returns the smaller of two numbers, which is all you need.

| Blank | What it should calculate |
|---|---|
| TASK 1 | The energy charged: the surplus, or the free space in the battery, whichever is smaller |
| TASK 2 | The energy discharged: the deficit, or the energy in the battery, whichever is smaller |
| TASK 3 | The grid import: whatever the battery could not cover |

Run the cell when you have filled the blanks. It only defines the function, so nothing is printed yet.

**Stuck?** Open one hint at a time. Try the first before you open the second.

<details><summary>TASK 1, hint 1</summary>

The battery can take the surplus, but never more than its free space. The free space is the capacity minus what is already stored: `capacity_kwh - soc`.
</details>

<details><summary>TASK 1, hint 2</summary>

`charge = min(surplus, ...)`. Fill in the free space.
</details>

<details><summary>TASK 2, hint 1</summary>

The battery can cover the deficit, but never more than it holds. What it holds is `soc`.
</details>

<details><summary>TASK 2, hint 2</summary>

`discharge = min(deficit, ...)`. Fill in what the battery holds.
</details>

<details><summary>TASK 3, hint 1</summary>

Whatever part of the deficit the battery did not cover must come from the grid.
</details>

<details><summary>TASK 3, hint 2</summary>

`grid_import[t] = deficit - ...`. Fill in what the battery delivered this hour.
</details>

**After you change a blank, run this cell again**, then run the test in the next section. The test uses the version of the function that was last run.

In [ ]:
def simulate_battery(pv_kwh, demand_kwh, capacity_kwh):
    # Simulate a battery hour by hour. All energies are in kWh per hour.
    #
    # Idealised battery: lossless, and able to charge or discharge its full
    # capacity within one hour. Section 4.6 removes both simplifications.
    # The battery follows a simple greedy rule: store any surplus, use the
    # store whenever there is a deficit.
    pv = pv_kwh.to_numpy()
    demand = demand_kwh.to_numpy()
    n_hours = len(pv)

    pv_direct = np.minimum(pv, demand)
    battery_discharge = np.zeros(n_hours)
    grid_import = np.zeros(n_hours)
    grid_export = np.zeros(n_hours)
    state_of_charge = np.zeros(n_hours)

    soc = 0.0  # the battery starts empty
    for t in range(n_hours):
        surplus = pv[t] - pv_direct[t]       # spare solar after serving demand
        deficit = demand[t] - pv_direct[t]   # demand still to be met

        if surplus > 0:
            # Spare solar: charge the battery, but only up to its free space.
            charge = min(surplus, capacity_kwh - soc)     # TASK 1
            soc += charge
            grid_export[t] = surplus - charge
        else:
            # Not enough solar: use the battery first, but only what it holds.
            discharge = min(deficit, soc)                 # TASK 2
            soc -= discharge
            battery_discharge[t] = discharge
            grid_import[t] = deficit - discharge          # TASK 3

        state_of_charge[t] = soc

    return pd.DataFrame({
        "pv_direct": pv_direct,
        "battery_discharge": battery_discharge,
        "grid_import": grid_import,
        "grid_export": grid_export,
        "soc": state_of_charge,
    }, index=pv_kwh.index)

### 4.4 Step 3: test your rule on four hours

Professional modellers test a model on a small case they can check by hand before trusting it on a full year. The four hours below use a 2 kWh battery:

| Hour | Solar | Demand | What should happen |
|---|---|---|---|
| 1 | 3.0 | 1.0 | 2.0 surplus fills the battery exactly |
| 2 | 2.0 | 0.5 | 1.5 surplus, battery full, so 1.5 is exported |
| 3 | 0.0 | 1.5 | 1.5 from the battery, 0.5 left in it |
| 4 | 0.0 | 1.0 | 0.5 from the battery, 0.5 imported |

Run the cell. If you see `NameError: name '____' is not defined`, a blank is still empty. If a check fails, compare your blank with the table above.

In [ ]:
test_index = pd.RangeIndex(4)
test = simulate_battery(pd.Series([3.0, 2.0, 0.0, 0.0], index=test_index),
                        pd.Series([1.0, 0.5, 1.5, 1.0], index=test_index),
                        capacity_kwh=2.0)
display(test)

expected = {
    "grid_export": [0.0, 1.5, 0.0, 0.0],
    "battery_discharge": [0.0, 0.0, 1.5, 0.5],
    "grid_import": [0.0, 0.0, 0.0, 0.5],
    "soc": [2.0, 2.0, 0.5, 0.0],
}
all_passed = True
for column, values in expected.items():
    passed = np.allclose(test[column], values)
    all_passed &= passed
    print(f"{'PASS' if passed else 'FAIL'}  {column}")

print("\nAll tests passed. Your battery rule works." if all_passed
      else "\nAt least one test failed. Check your blanks against the table above.")

### 4.5 Step 4: run the full year and compare

Now run the idealised model for all 8,760 hours with a 5 kWh battery and compare it with the baseline. Two checks run first. The **energy balance check** confirms that no energy has been created or lost: everything the home used must have come from solar, the battery or the grid. The **limits check** confirms that the battery never holds more than its capacity or less than nothing.

**Before you run the cell, predict.** Without a battery the self-sufficiency rate is shown in section 4.2. What will it be with a 5 kWh battery? The Reporter enters the group's guess in the Google Form (question 2), then you run the cell and compare.

In [ ]:
with_battery = simulate_battery(home["pv_kwh"], home["demand_kwh"], BATTERY_KWH)

served = with_battery[["pv_direct", "battery_discharge", "grid_import"]].sum(axis=1)
assert np.allclose(served, home["demand_kwh"]), "Energy balance failed: check your blanks."
assert with_battery["soc"].max() <= BATTERY_KWH + 1e-9, "The battery holds more than its capacity: check TASK 1."
assert with_battery["soc"].min() >= -1e-9, "The battery went below empty: check TASK 2."
print("Checks passed: every kWh of demand is accounted for and the battery stays within its limits.\n")

results = pd.concat([
    baseline,
    summarise(f"{BATTERY_KWH:g} kWh battery, idealised", home["pv_kwh"], home["demand_kwh"],
              with_battery["grid_import"], with_battery["grid_export"]),
], axis=1)
results["Change"] = results.iloc[:, 1] - results.iloc[:, 0]

show_table(results, 1)
save_table(results, "tab02_task1_results")

### 4.6 A more realistic battery

The idealised result is an **upper bound**. A real home battery differs in two important ways:

- **Round-trip efficiency.** Some energy is lost as heat in the inverter and the cells. The AC to AC round-trip efficiency, the energy you get back divided by the energy you put in, is about 85 to 93 % for current home batteries. We use 90 % and split the loss equally between charging and discharging, so each direction has an efficiency of $\sqrt{0.90} \approx 0.949$.
- **Power limit.** A typical home battery charges and discharges at no more than about 3 kW, so it cannot take a large surplus or meet a large deficit in one hour.

`BATTERY_KWH` is the **usable** capacity, so the battery can be used from empty to full. Section 3.1 sets all three values.

The state of charge changes with each step as follows, where $\eta = \sqrt{\text{round-trip efficiency}}$:

$$SOC_{t} = SOC_{t-1} + \eta \, \text{Charge}_t - \frac{\text{Discharge}_t}{\eta}$$

This function is complete, so you only need to run it. It still uses a simple greedy rule, not an optimised schedule, and it still uses hourly steps.

In [ ]:
def simulate_battery_realistic(pv_kwh, demand_kwh, capacity_kwh, power_kw,
                               round_trip_efficiency, dt_h=1.0):
    # Battery with a power limit and losses. Energies in kWh per time step.
    # Charge and discharge are measured on the AC side (at the home's meter).
    eta = np.sqrt(round_trip_efficiency)   # one-way efficiency
    max_step = power_kw * dt_h             # most energy that can move in one step, kWh

    pv = pv_kwh.to_numpy()
    demand = demand_kwh.to_numpy()
    n_steps = len(pv)

    pv_direct = np.minimum(pv, demand)
    battery_charge = np.zeros(n_steps)
    battery_discharge = np.zeros(n_steps)
    grid_import = np.zeros(n_steps)
    grid_export = np.zeros(n_steps)
    state_of_charge = np.zeros(n_steps)

    soc = 0.0
    for t in range(n_steps):
        surplus = pv[t] - pv_direct[t]
        deficit = demand[t] - pv_direct[t]
        if surplus > 0:
            charge = min(surplus, max_step, (capacity_kwh - soc) / eta)
            soc += charge * eta
            battery_charge[t] = charge
            grid_export[t] = surplus - charge
        else:
            discharge = min(deficit, max_step, soc * eta)
            soc -= discharge / eta
            battery_discharge[t] = discharge
            grid_import[t] = deficit - discharge
        state_of_charge[t] = soc

    return pd.DataFrame({
        "pv_direct": pv_direct,
        "battery_charge": battery_charge,
        "battery_discharge": battery_discharge,
        "grid_import": grid_import,
        "grid_export": grid_export,
        "soc": state_of_charge,
    }, index=pv_kwh.index)


realistic = simulate_battery_realistic(home["pv_kwh"], home["demand_kwh"], BATTERY_KWH,
                                       BATTERY_POWER_KW, ROUND_TRIP_EFFICIENCY)

# Verification: every kWh of solar and of demand is accounted for, and limits hold.
battery_losses = realistic["battery_charge"].sum() - realistic["battery_discharge"].sum() - realistic["soc"].iloc[-1]
assert np.allclose(realistic[["pv_direct", "battery_charge", "grid_export"]].sum(axis=1), home["pv_kwh"]), "Solar balance failed."
assert np.allclose(realistic[["pv_direct", "battery_discharge", "grid_import"]].sum(axis=1), home["demand_kwh"]), "Demand balance failed."
assert battery_losses >= 0, "The battery created energy."
assert realistic["soc"].between(-1e-9, BATTERY_KWH + 1e-9).all(), "State of charge out of range."
assert (realistic[["battery_charge", "battery_discharge"]].max() <= BATTERY_POWER_KW + 1e-9).all(), "Power limit exceeded."
print("Checks passed: solar and demand balance, losses are positive, and the limits hold.\n")

# Compare with the baseline and, if Step 4 has run, with the idealised battery.
scenarios = {"No battery": (no_battery, 0.0)}
if "with_battery" in globals():
    scenarios["Idealised battery"] = (with_battery, 0.0)
scenarios["Realistic battery"] = (realistic, battery_losses)

comparison = pd.concat([
    pd.concat([
        summarise(name, home["pv_kwh"], home["demand_kwh"], flows["grid_import"], flows["grid_export"]),
        pd.Series({"Battery losses (kWh)": losses,
                   "Equivalent full cycles per year": flows["battery_discharge"].sum() / BATTERY_KWH}),
    ]).rename(name)
    for name, (flows, losses) in scenarios.items()
], axis=1)

show_table(comparison, 1)
save_table(comparison, "tab03_idealised_vs_realistic_battery")

**Read the table.** The realistic battery gives a slightly lower self-sufficiency rate than the idealised one, mainly because of the round-trip losses. The 3 kW limit matters less here, because a 4 kWp system rarely has more than 3 kW of surplus for a whole hour. The idealised figure is an upper bound. Both results still use a smooth demand profile, which is a larger source of optimism than the battery assumptions (see the limitations in section 6). **Equivalent full cycles** are the energy discharged in a year divided by the usable capacity. The number of cycles matters because battery warranties and lifetime are often stated in cycles as well as years.

From here on, the figures use the realistic battery.

### 4.7 What the battery does on a sunny day and a dull day

**Figure 3** shows how demand is met hour by hour on a sunny summer day and a winter day, using the realistic battery. The stacked areas add up to demand. The lower panels show the battery's state of charge.

Both days are chosen automatically from the data: the June day with the most solar generation (a best case) and the December day with the least (a worst case).

In [ ]:
june = home.loc["2019-06", "pv_kwh"].resample("D").sum()
SUMMER_DAY = june.idxmax().strftime("%Y-%m-%d")
december = home.loc["2019-12", "pv_kwh"].resample("D").sum()
WINTER_DAY = december.idxmin().strftime("%Y-%m-%d")


def daily_self_sufficiency(day, flows):
    d = home.loc[day, "demand_kwh"].sum()
    return 100 * (d - flows.loc[day, "grid_import"].sum()) / d


fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex="col",
                         gridspec_kw={"height_ratios": [3, 1]})
for col, (label, day) in enumerate([("Sunny summer day", SUMMER_DAY), ("Dull winter day", WINTER_DAY)]):
    flows = realistic.loc[day]
    hours = flows.index.hour
    ax = axes[0, col]
    ax.stackplot(hours, flows["pv_direct"], flows["battery_discharge"], flows["grid_import"],
                 colors=[COLOURS["solar"], COLOURS["battery"], COLOURS["grid"]],
                 labels=["Solar used directly", "Battery discharge", "Grid import"], alpha=0.9)
    ax.plot(hours, home.loc[day, "pv_kwh"], color=COLOURS["solar"], linestyle="--",
            linewidth=1.2, label="Solar generation")
    ax.set_title(f"{label}, {pd.Timestamp(day):%d %B %Y}\n"
                 f"Self-sufficiency rate: {daily_self_sufficiency(day, no_battery):.0f} % without battery, "
                 f"{daily_self_sufficiency(day, realistic):.0f} % with battery", loc="left")
    ax.set_ylabel("Energy per hour (kWh)")
    axes[1, col].fill_between(hours, flows["soc"], color=COLOURS["battery"], alpha=0.35, linewidth=0)
    axes[1, col].plot(hours, flows["soc"], color=COLOURS["battery"], linewidth=1.2)
    axes[1, col].set_ylim(0, BATTERY_KWH * 1.05)
    axes[1, col].set_ylabel("Stored (kWh)")
    axes[1, col].set_xlabel("Hour of day (UTC; UK clock is UTC + 1)" if col == 0 else "Hour of day (UTC, same as UK clock)")
    axes[1, col].set_xticks(range(0, 24, 3))
axes[0, 0].legend(loc="upper left", fontsize=8)
fig.suptitle(f"Figure 3. How demand is met with a {BATTERY_KWH:g} kWh, {BATTERY_POWER_KW:g} kW battery (realistic model)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save_figure(fig, "fig03_battery_daily_dispatch")
plt.show()

daily = pd.DataFrame({
    "Self-sufficiency rate without battery (%)": [daily_self_sufficiency(d, no_battery) for d in (SUMMER_DAY, WINTER_DAY)],
    "Self-sufficiency rate with realistic battery (%)": [daily_self_sufficiency(d, realistic) for d in (SUMMER_DAY, WINTER_DAY)],
}, index=[f"Summer, {SUMMER_DAY}", f"Winter, {WINTER_DAY}"])
show_table(daily, 0)
save_table(daily, "tab04_daily_self_sufficiency")

**Figure 4** repeats the comparison for every month. Ask yourself why the self-sufficiency rate stays well below 100 % in winter, even with the battery, while in summer the battery can cover almost all of the demand.

In [ ]:
def monthly_self_sufficiency(flows):
    m = pd.DataFrame({"demand": home["demand_kwh"], "import": flows["grid_import"]}).resample("MS").sum()
    return 100 * (m["demand"] - m["import"]) / m["demand"]


monthly_ss = pd.DataFrame({
    "Without battery (%)": monthly_self_sufficiency(no_battery),
    f"With {BATTERY_KWH:g} kWh realistic battery (%)": monthly_self_sufficiency(realistic),
})
monthly_ss.index = monthly_ss.index.strftime("%b")

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(12)
ax.bar(x - 0.2, monthly_ss.iloc[:, 0], width=0.4, color=COLOURS["export"], label="Without battery")
ax.bar(x + 0.2, monthly_ss.iloc[:, 1], width=0.4, color=COLOURS["battery"], label=f"With {BATTERY_KWH:g} kWh realistic battery")
ax.set_xticks(x, monthly_ss.index)
ax.set_ylim(0, 118)
ax.set_yticks(range(0, 101, 20))
ax.set_ylabel("Self-sufficiency rate (%)")
ax.set_title("Figure 4. Monthly self-sufficiency rate", loc="left")
ax.legend(loc="upper left", ncol=2, bbox_to_anchor=(0, 1.0))
ax.grid(axis="x", visible=False)
fig.tight_layout()
save_figure(fig, "fig04_monthly_self_sufficiency")
plt.show()

show_table(monthly_ss, 0)
save_table(monthly_ss, "tab05_monthly_self_sufficiency")

### 4.8 Group scenario: what if the home were different?

Your **group number** decides your scenario. Each scenario changes **one** thing about the home, so every group gets a different result to compare in the debrief.

| Group | Task 1 scenario | Task 2 scenario (section 5.6) |
|---|---|---|
| 1 | A | P |
| 2 | B | Q |
| 3 | C | R |
| 4 | D | S |
| 5 | E | T |
| 6 | F | R |
| 7 | G | T |
| 8 | B | S |

Groups 9 and above start again from group 1 (group 9 has the scenarios of group 1).

The scenarios use the **realistic** battery from section 4.6, so the reference home here matches the realistic result, not the idealised one from section 4.5.

1. **Predict first.** Find your scenario below. Agree as a group whether the self-sufficiency rate will go up, stay the same or go down compared with the reference home, and roughly by how much, **before** you run the cell.
2. Type your group number in the first line of the cell and run it.
3. **Explain.** Was your prediction right? Use the numbers to say why.

**Reporter:** from the column for your scenario, enter the **Self-sufficiency rate, with battery** and the **Annual saving from the battery** in the Google Form (questions 3 and 4).

| Scenario | What changes | Reference value |
|---|---|---|
| A | 6 kWp of solar (a larger roof) | 4 kWp |
| B | 2.5 kWp of solar (a small roof) | 4 kWp |
| C | Demand of 3,800 kWh a year (Ofgem high TDCV) | 2,500 kWh |
| D | Demand of 1,600 kWh a year (Ofgem low TDCV) | 2,500 kWh |
| E | A 10 kWh battery | 5 kWh |
| F | An older battery: 85 % round-trip efficiency, 2.5 kW | 90 %, 3 kW |
| G | A low export tariff of 4 p/kWh | 12 p/kWh |

Larger or smaller solar systems are modelled by scaling the PVGIS output, and other demand levels by scaling the same profile. Both are reasonable first approximations.

In [ ]:
MY_GROUP = 1   # TASK: type your group number, then run this cell.

# Scenario letters for each group: (Task 1, Task 2). Groups above 8 wrap round.
GROUP_SCENARIOS = {1: ("A", "P"), 2: ("B", "Q"), 3: ("C", "R"), 4: ("D", "S"),
                   5: ("E", "T"), 6: ("F", "R"), 7: ("G", "T"), 8: ("B", "S")}


def scenarios_for_group(group):
    # Return the (Task 1, Task 2) scenario letters for a group number.
    if not isinstance(group, int) or group < 1:
        raise ValueError(f"MY_GROUP must be a whole number such as 3, not {group!r}.")
    return GROUP_SCENARIOS[(group - 1) % len(GROUP_SCENARIOS) + 1]


TASK1_SCENARIOS = {
    "A": {"label": "6 kWp of solar", "pv_kw": 6.0},
    "B": {"label": "2.5 kWp of solar", "pv_kw": 2.5},
    "C": {"label": "3,800 kWh demand", "demand_kwh": 3800},
    "D": {"label": "1,600 kWh demand", "demand_kwh": 1600},
    "E": {"label": "10 kWh battery", "battery_kwh": 10.0},
    "F": {"label": "older battery, 85 %, 2.5 kW", "efficiency": 0.85, "power_kw": 2.5},
    "G": {"label": "export at 4 p/kWh", "export_price": 0.04},
}


def run_home_scenario(changes):
    # Run the realistic model with no battery and with a battery for one set of changes.
    pv_kwh = home["pv_kwh"] * changes.get("pv_kw", PV_PEAK_KW) / PV_PEAK_KW
    demand_kwh = home["demand_kwh"] * changes.get("demand_kwh", ANNUAL_DEMAND_KWH) / ANNUAL_DEMAND_KWH
    export_price = changes.get("export_price", EXPORT_PRICE)
    flows = simulate_battery_realistic(pv_kwh, demand_kwh,
                                       changes.get("battery_kwh", BATTERY_KWH),
                                       changes.get("power_kw", BATTERY_POWER_KW),
                                       changes.get("efficiency", ROUND_TRIP_EFFICIENCY))
    no_batt = summarise("no battery", pv_kwh, demand_kwh,
                        (demand_kwh - pv_kwh).clip(lower=0), (pv_kwh - demand_kwh).clip(lower=0),
                        export_price=export_price)
    with_batt = summarise("battery", pv_kwh, demand_kwh, flows["grid_import"], flows["grid_export"],
                          export_price=export_price)
    return pd.Series({
        "Self-sufficiency rate, no battery (%)": no_batt["Self-sufficiency rate (%)"],
        "Self-sufficiency rate, with battery (%)": with_batt["Self-sufficiency rate (%)"],
        "Self-consumption rate, with battery (%)": with_batt["Self-consumption rate (%)"],
        "Net energy cost, with battery (GBP/yr)": with_batt["Net energy cost (GBP/yr)"],
        "Annual saving from the battery (GBP/yr)": no_batt["Net energy cost (GBP/yr)"] - with_batt["Net energy cost (GBP/yr)"],
    })


letter1 = scenarios_for_group(MY_GROUP)[0]
print(f"Group {MY_GROUP}: Task 1 scenario {letter1}, {TASK1_SCENARIOS[letter1]['label']}\n")

scenario_table = pd.concat({
    "Reference home": run_home_scenario({}),
    f"Scenario {letter1}: {TASK1_SCENARIOS[letter1]['label']}": run_home_scenario(TASK1_SCENARIOS[letter1]),
}, axis=1)
scenario_table["Difference"] = scenario_table.iloc[:, 1] - scenario_table.iloc[:, 0]

show_table(scenario_table, 1)
save_table(scenario_table, f"tab06_task1_scenario_{letter1}")

**Instructor only: answer key for all scenarios.** This cell is not in the student notebook. Run it before the session to know what each group should report.

In [ ]:
key1 = pd.DataFrame({f"{k}: {v['label']}": run_home_scenario(v) for k, v in TASK1_SCENARIOS.items()}).T
key1.loc["Reference home"] = run_home_scenario({})
show_table(key1, 1)
save_table(key1, "instructor_key_task1_scenarios")

### 4.9 Stretch task: how big should the battery be?

If your group finishes early, run the cell below. It repeats the full-year simulation with the realistic battery for usable capacities from 0 to 15 kWh, keeping the 3 kW power limit and 90 % round-trip efficiency.

Look at the shape of the curve in **Figure 5**. The first few kWh are used almost every day. Each extra kWh is only useful on days with a large surplus and a large evening demand, so its value falls. This is **diminishing returns**, and it is why the most cost-effective battery is rarely the largest one. Compare the extra saving from each kWh with the cost of buying it, typically several hundred pounds per kWh installed, and the case for a very large battery on savings alone is weak.

To go further, change `ROUND_TRIP_EFFICIENCY` or `BATTERY_POWER_KW` in section 3.1, run the notebook again and see how the results change.

In [ ]:
sizes = np.arange(0, 16, 1)
sweep = pd.DataFrame([
    summarise(size, home["pv_kwh"], home["demand_kwh"], flows["grid_import"], flows["grid_export"])
    for size in sizes
    for flows in [simulate_battery_realistic(home["pv_kwh"], home["demand_kwh"], size,
                                             BATTERY_POWER_KW, ROUND_TRIP_EFFICIENCY)]
])
sweep.index.name = "Usable battery capacity (kWh)"
sweep["Annual saving vs no battery (GBP)"] = sweep["Net energy cost (GBP/yr)"].iloc[0] - sweep["Net energy cost (GBP/yr)"]
sweep["Extra saving from the last kWh (GBP)"] = sweep["Annual saving vs no battery (GBP)"].diff().fillna(0)

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(sweep.index, sweep["Self-sufficiency rate (%)"], color=COLOURS["battery"], marker="o", linewidth=1.8)
ax1.set_xlabel("Usable battery capacity (kWh)")
ax1.set_ylabel("Self-sufficiency rate (%)", color=COLOURS["battery"])
ax2 = ax1.twinx()
ax2.bar(sweep.index, sweep["Extra saving from the last kWh (GBP)"], color=COLOURS["grid"], alpha=0.35, width=0.6)
ax2.set_ylabel("Extra annual saving from each kWh (GBP)", color=COLOURS["grid"])
ax2.grid(False)
ax2.spines["right"].set_visible(True)
ax1.set_title("Figure 5. Diminishing returns from larger batteries", loc="left")
fig.tight_layout()
save_figure(fig, "fig05_battery_size_sweep")
plt.show()

show_table(sweep, 1)
save_table(sweep, "tab07_battery_size_sweep")

**Discuss in your group (two minutes):**

1. Why is the self-sufficiency rate so much higher in summer than in winter, even with the battery?
2. The demand profile is smooth and the time step is one hour. Does that make the results too optimistic or too pessimistic, and why?
3. What would change if the home had an electric vehicle, a heat pump or a time-of-use tariff?

## 5. Task 2: the merit order and a carbon price

**Question.** What does a carbon price do to the electricity price and to emissions?

**Time:** 10 minutes. Rotate the roles before you start, so a new person is the Modeller.

### 5.1 The idea

Power stations are ranked by their **short-run marginal cost (SRMC)**, the cost of producing one more MWh: mainly fuel, variable operation and maintenance, and carbon. This ranking is the **merit order**. The least-cost way to meet demand uses the cheapest units first. The last unit needed is the **marginal unit**, and its SRMC sets the **clearing price** for that hour.

The model is a **uniform-price** market: every generator running in an hour receives the same clearing price. This is a standard simplification of a day-ahead auction, such as those in continental Europe. Great Britain works differently in detail: most electricity is traded in bilateral contracts and on power exchanges under the BETTA arrangements, and the National Energy System Operator (NESO) then balances the system through the pay-as-bid Balancing Mechanism. The merit order is still the best simple guide to how prices are formed.

A carbon price adds a cost to every tonne of CO2 emitted. For each power station:

$$\text{SRMC} \; (\text{GBP/MWh}) = \text{SRMC excluding carbon} \; (\text{GBP/MWh}) + \text{carbon price} \; (\text{GBP/tCO}_2) \times \text{emission intensity} \; (\text{tCO}_2\text{/MWh})$$

Coal emits more than twice as much CO2 per MWh as gas, so a high enough carbon price moves coal behind gas in the merit order. This is called **coal-to-gas fuel switching**.

> **Note on coal.** Great Britain's last coal power station, Ratcliffe-on-Soar, stopped generating at 15:35 on 30 September 2024. Coal is a hypothetical unit here. It is included because it makes the effect of a carbon price easy to see, and because it is still used in other European systems.

### 5.2 The power system

The system is a stylised one-day snapshot of a windy winter day, loosely resembling Great Britain in the late 2010s, when coal still ran. All values are **illustrative teaching assumptions**, not market data.

| Assumption | Value | Basis |
|---|---|---|
| Gas (CCGT) emission intensity | 0.37 tCO2/MWh | Median direct emissions for a combined-cycle gas turbine (IPCC, 2014) |
| Coal emission intensity | 0.90 tCO2/MWh | An older subcritical coal plant at about 38 % efficiency, with the IPCC default emission factor for bituminous coal, 94.6 tCO2/TJ (IPCC, 2006). The IPCC median for modern pulverised coal is lower, 0.76 tCO2/MWh |
| Gas SRMC excluding carbon | 60 GBP/MWh | Consistent with 2025 wholesale gas prices. In October 2026 gas was much more expensive, so a current figure would be higher |
| Coal SRMC excluding carbon | 35 GBP/MWh | Illustrative |
| Wind | 20 GW, available 47 to 75 % | `WIND_AVAILABILITY` is the hourly availability, called `p_max_pu` in PyPSA: the share of installed capacity the weather allows in that hour. It is not the annual capacity factor |

Emission intensities are **direct CO2** from burning the fuel. They exclude upstream emissions, such as methane leaks from gas supply, and other greenhouse gases, so they are tCO2, not tCO2e.

In [ ]:
HOURS = pd.RangeIndex(24, name="hour")

# Hourly demand, MW. Peaks at 45,000 MW at 18:00.
DEMAND_MW = pd.Series([
    30.0, 28.5, 27.5, 27.0, 27.0, 28.0, 31.0, 35.0, 38.0, 39.0, 39.5, 39.5,
    39.0, 38.5, 38.5, 39.5, 42.0, 44.5, 45.0, 44.0, 42.0, 39.0, 35.0, 32.0,
], index=HOURS) * 1000

# Hourly wind availability (PyPSA p_max_pu): share of installed capacity the weather allows.
WIND_AVAILABILITY = pd.Series([
    0.70, 0.72, 0.72, 0.71, 0.70, 0.68, 0.64, 0.60, 0.56, 0.52, 0.50, 0.48,
    0.47, 0.47, 0.48, 0.52, 0.58, 0.66, 0.75, 0.74, 0.72, 0.70, 0.70, 0.70,
], index=HOURS)

GENERATORS = pd.DataFrame({
    "capacity_mw":           [20000, 8000, 10000, 25000],
    "srmc_excl_carbon_gbp_mwh": [0,     10,   35,    60],
    "emission_intensity_tco2_mwh":    [0.00,  0.00, 0.90,  0.37],
}, index=pd.Index(["Wind", "Nuclear", "Coal", "Gas"], name="technology"))

show_table(GENERATORS)
save_table(GENERATORS, "tab08_generator_assumptions")

### 5.3 Build and solve the model in PyPSA

PyPSA describes an energy system as a set of **components**: buses (connection points), loads, generators, storage and lines. The function below:

1. creates a network with one bus, so transmission limits are ignored;
2. adds the demand and the four generators, each with its SRMC including the carbon price;
3. solves a linear optimisation, an **economic dispatch**, with the open-source solver HiGHS, which finds the cheapest way to meet demand in every hour;
4. returns the network, so we can read the results from it.

**Where does the price come from?** The optimisation has one constraint per hour: generation must equal demand. Its **dual value** (shadow price) is the cost of meeting one more MWh of demand in that hour, which is the marginal unit's SRMC. PyPSA reports it as `n.buses_t.marginal_price`. When demand falls exactly at the boundary between two units, the dual is not unique and the solver may report either unit's cost. This is called **degeneracy**, and it is a property of the mathematics, not an error.

The model reports two different money totals. **Total operating cost** is what the optimisation minimises: the SRMC of every MWh generated, including the carbon cost. **Consumer payments** are the clearing price times demand in every hour. Payments are larger, because cheaper units also receive the clearing price, and the difference, called **inframarginal rent**, helps them recover their fixed costs.

You do not need to change this cell. Read the comments to see how each line maps to the steps above.

In [ ]:
def build_and_solve(carbon_price, generators=None):
    # Build the one-bus dispatch model for a given carbon price and solve it.
    # `generators` lets a scenario change the fleet; by default GENERATORS is used.
    generators = GENERATORS if generators is None else generators
    n = pypsa.Network()
    n.set_snapshots(HOURS)

    n.add("Carrier", "AC")
    n.add("Carrier", generators.index, co2_emissions=generators["emission_intensity_tco2_mwh"])
    n.add("Bus", "GB", carrier="AC")
    n.add("Load", "Demand", bus="GB", p_set=DEMAND_MW)

    for tech, g in generators.iterrows():
        n.add(
            "Generator", tech,
            bus="GB",
            carrier=tech,
            p_nom=g["capacity_mw"],
            p_max_pu=WIND_AVAILABILITY if tech == "Wind" else 1.0,
            marginal_cost=g["srmc_excl_carbon_gbp_mwh"] + carbon_price * g["emission_intensity_tco2_mwh"],
        )

    status, condition = n.optimize(solver_name="highs", solver_options={"output_flag": False})
    if status != "ok" or condition != "optimal":
        raise RuntimeError(f"The optimisation did not solve to optimality: {status}, {condition}")
    return n


def summarise_run(n, carbon_price):
    # Collect the headline results of one model run.
    dispatch = n.generators_t.p.clip(lower=0)
    price = n.buses_t.marginal_price["GB"]
    peak_hour = DEMAND_MW.idxmax()
    emissions = (dispatch * GENERATORS["emission_intensity_tco2_mwh"]).sum().sum()
    return {
        "Carbon price (GBP/tCO2)": carbon_price,
        "Peak-hour price (GBP/MWh)": price[peak_hour],
        "Price-setting unit at peak": marginal_unit(n, peak_hour),
        "Total operating cost incl. carbon (GBP m)": n.objective / 1e6,
        "of which carbon cost (GBP m)": carbon_price * emissions / 1e6,
        "Consumer payments (GBP m)": (price * DEMAND_MW).sum() / 1e6,
        "Direct CO2 emissions (tCO2)": emissions,
        "Coal generation (MWh)": dispatch["Coal"].sum(),
        "Gas generation (MWh)": dispatch["Gas"].sum(),
    }


def marginal_unit(n, hour):
    # The marginal unit is running but not at its maximum output.
    output = n.generators_t.p.loc[hour]
    available = n.generators.p_nom * n.get_switchable_as_dense("Generator", "p_max_pu").loc[hour]
    part_loaded = (output > 1) & (output < available - 1)
    return ", ".join(output.index[part_loaded]) or "none"


run_log = []  # every run is added here, so you can compare scenarios
print("Model functions ready.")

### 5.4 TASK: run the model with and without a carbon price

1. Run the cell below as it is, with `CARBON_PRICE = 0`.
2. Record the peak-hour price, total operating cost, consumer payments and direct CO2 emissions.
3. Change **only** the first line to `CARBON_PRICE = 80` and run the cell again.
4. Say which unit sets the peak price, and why the price changed.

Every run is added to a results log, so the table grows each time you run the cell.

**Before step 1, predict.** At a carbon price of 80 GBP/tCO2, compared with no carbon price, will the peak-hour price and the emissions go up, stay the same or go down? The Reporter enters the group's prediction in the Google Form (question 5).

**Reporter:** after the run at 80, enter the peak-hour price and the direct CO2 emissions from row 80 of the results table in the Google Form (questions 6 and 7).

In [ ]:
CARBON_PRICE = 0   # GBP per tonne of CO2. TASK: change this to 80 and run again.
# For context: in October 2026 a GB power station paid roughly 60 to 65 GBP/t for UK ETS allowances
# plus the 18 GBP/t Carbon Price Support, about 80 GBP/t in total.

network = build_and_solve(CARBON_PRICE)
run_log.append(summarise_run(network, CARBON_PRICE))

log = pd.DataFrame(run_log).drop_duplicates("Carbon price (GBP/tCO2)", keep="last")
log = log.set_index("Carbon price (GBP/tCO2)").sort_index()
show_table(log, 1)
save_table(log, "tab09_task2_results_log")

### 5.5 See the result

**Figure 6** draws the merit order at the peak hour for the carbon price you just ran. Each block is one technology: its width is the capacity available and its height is the SRMC. The dashed line is demand, and the clearing price is where demand meets the supply curve.

**Figure 7** shows the optimised dispatch for the whole day and the hourly clearing price.

In [ ]:
def plot_merit_order(carbon_price, hour=None, ax=None):
    # Draw the supply curve (merit order) at one hour for a given carbon price.
    hour = DEMAND_MW.idxmax() if hour is None else hour
    supply = GENERATORS.copy()
    supply["available_mw"] = supply["capacity_mw"]
    supply.loc["Wind", "available_mw"] *= WIND_AVAILABILITY[hour]
    supply["marginal_cost"] = supply["srmc_excl_carbon_gbp_mwh"] + carbon_price * supply["emission_intensity_tco2_mwh"]
    supply = supply.sort_values("marginal_cost")
    supply["left_gw"] = supply["available_mw"].cumsum().shift(fill_value=0) / 1000

    demand_gw = DEMAND_MW[hour] / 1000
    marginal = supply[(supply["left_gw"] < demand_gw)].iloc[-1]
    ax = ax or plt.gca()
    for tech, s in supply.iterrows():
        width_gw = s["available_mw"] / 1000
        ax.bar(s["left_gw"], max(s["marginal_cost"], 0.8), width=width_gw, align="edge",
               color=COLOURS[tech.lower()], edgecolor="white", linewidth=1)
        label_x = s["left_gw"] + width_gw / 2
        if abs(label_x - demand_gw) < 3:      # keep the label clear of the demand line
            label_x = s["left_gw"] + width_gw * 0.25
        ax.text(label_x, max(s["marginal_cost"], 0.8) + 2,
                f"{tech}\n{s['marginal_cost']:.0f}", ha="center", va="bottom", fontsize=8)
    ax.axvline(demand_gw, color=COLOURS["demand"], linestyle="--", linewidth=1.2)
    y_top = max(125, supply["marginal_cost"].max() * 1.15)   # room for high carbon prices
    ax.text(demand_gw + 0.4, y_top * 0.94, f"Demand {demand_gw:.0f} GW", fontsize=8, va="top")
    ax.axhline(marginal["marginal_cost"], color=COLOURS["price"], linewidth=0.8, linestyle=":")
    ax.text(0.5, marginal["marginal_cost"] + 2, f"Clearing price {marginal['marginal_cost']:.1f} GBP/MWh",
            fontsize=8, fontweight="bold")
    ax.set_xlim(0, supply["available_mw"].sum() / 1000)
    ax.set_ylim(0, y_top)
    ax.set_xlabel("Cumulative available capacity (GW)")
    ax.set_ylabel("SRMC (GBP/MWh)")
    ax.set_title(f"Carbon price {carbon_price:g} GBP/tCO2, hour {hour}:00", loc="left")
    ax.grid(axis="x", visible=False)
    return ax


fig, ax = plt.subplots(figsize=(9, 4.5))
plot_merit_order(CARBON_PRICE, ax=ax)
fig.suptitle("Figure 6. Merit order at the peak hour", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save_figure(fig, f"fig06_merit_order_carbon_{CARBON_PRICE:g}")
plt.show()

dispatch = network.generators_t.p.clip(lower=0) / 1000
price = network.buses_t.marginal_price["GB"]

fig, ax = plt.subplots(figsize=(10, 4.5))
order = ["Nuclear", "Wind", "Coal", "Gas"]
ax.stackplot(dispatch.index, [dispatch[t] for t in order],
             colors=[COLOURS[t.lower()] for t in order], labels=order, alpha=0.9)
ax.plot(DEMAND_MW.index, DEMAND_MW / 1000, color=COLOURS["demand"], linewidth=1.5, label="Demand")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Generation (GW)")
ax.set_xticks(range(0, 24, 2))
ax.set_xlim(0, 23)
price_ax = ax.twinx()
price_ax.step(price.index, price, where="mid", color=COLOURS["price"], linestyle="--", linewidth=1.3, label="Clearing price")
price_ax.set_ylabel("Clearing price (GBP/MWh)")
price_ax.set_ylim(0, max(125, price.max() * 1.15))
price_ax.grid(False)
price_ax.spines["right"].set_visible(True)
handles = ax.get_legend_handles_labels()[0] + price_ax.get_legend_handles_labels()[0]
labels = ax.get_legend_handles_labels()[1] + price_ax.get_legend_handles_labels()[1]
ax.legend(handles, labels, loc="upper left", ncol=3, fontsize=8)
ax.set_title(f"Carbon price {CARBON_PRICE:g} GBP/tCO2", loc="left")
fig.suptitle("Figure 7. Optimised dispatch and clearing price over one day", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save_figure(fig, f"fig07_dispatch_carbon_{CARBON_PRICE:g}")
plt.show()

hourly = dispatch.round(2).assign(**{"Demand (GW)": DEMAND_MW / 1000, "Clearing price (GBP/MWh)": price.round(2)})
save_table(hourly, f"tab10_hourly_dispatch_carbon_{CARBON_PRICE:g}")

### 5.6 Group scenario: change one thing about the power system

Your group's Task 2 scenario is in the table in section 4.8. Each scenario changes one thing compared with the **reference case, a carbon price of 80 GBP/tCO2**.

1. **Predict first.** Agree as a group whether the peak-hour price and the emissions will go up, stay the same or go down.
2. Run the cell. It uses the `MY_GROUP` number you set in section 4.8.
3. **Explain** the result using the merit order: which unit sets the price, and did coal and gas swap places?

| Scenario | What changes | Reference value |
|---|---|---|
| P | Carbon price of 40 GBP/tCO2 | 80 |
| Q | Carbon price of 120 GBP/tCO2 | 80 |
| R | Gas SRMC excluding carbon of 140 GBP/MWh, closer to wholesale gas prices in autumn 2026 | 60 |
| S | 30 GW of wind instead of 20 GW | 20 GW |
| T | A calm day: wind output halved in every hour | 20 GW at the normal availability |

**Reporter:** enter your scenario's peak-hour price and direct CO2 emissions in the Google Form (questions 8 and 9), then one sentence explaining either of your scenario results, Task 1 or Task 2 (question 10), and submit.

In [ ]:
# Uses MY_GROUP from section 4.8.
if "scenarios_for_group" not in globals():
    raise NameError("Run the code cell in section 4.8 first: that is where you set MY_GROUP.")

REFERENCE_CARBON_PRICE = 80
TASK2_SCENARIOS = {
    "P": {"label": "carbon price 40", "carbon_price": 40},
    "Q": {"label": "carbon price 120", "carbon_price": 120},
    "R": {"label": "gas SRMC 140 GBP/MWh", "gas_srmc": 140},
    "S": {"label": "30 GW of wind", "wind_mw": 30000},
    "T": {"label": "calm day, wind halved", "wind_mw": 10000},   # same as halving availability
}


def run_system_scenario(changes):
    # Solve the dispatch model with one change from the reference case.
    fleet = GENERATORS.copy()
    if "gas_srmc" in changes:
        fleet.loc["Gas", "srmc_excl_carbon_gbp_mwh"] = changes["gas_srmc"]
    if "wind_mw" in changes:
        fleet.loc["Wind", "capacity_mw"] = changes["wind_mw"]
    carbon_price = changes.get("carbon_price", REFERENCE_CARBON_PRICE)
    result = summarise_run(build_and_solve(carbon_price, fleet), carbon_price)
    keep = ["Carbon price (GBP/tCO2)", "Peak-hour price (GBP/MWh)", "Price-setting unit at peak",
            "Consumer payments (GBP m)", "Direct CO2 emissions (tCO2)",
            "Coal generation (MWh)", "Gas generation (MWh)"]
    return pd.Series({k: round(result[k], 1) if isinstance(result[k], float) else result[k] for k in keep})


letter2 = scenarios_for_group(MY_GROUP)[1]
print(f"Group {MY_GROUP}: Task 2 scenario {letter2}, {TASK2_SCENARIOS[letter2]['label']}\n")

scenario2_table = pd.concat({
    "Reference (80 GBP/tCO2)": run_system_scenario({}),
    f"Scenario {letter2}: {TASK2_SCENARIOS[letter2]['label']}": run_system_scenario(TASK2_SCENARIOS[letter2]),
}, axis=1)
show_table(scenario2_table, 1)
save_table(scenario2_table, f"tab11_task2_scenario_{letter2}")

**Instructor only: answer key for all Task 2 scenarios.**

In [ ]:
key2 = pd.DataFrame({f"{k}: {v['label']}": run_system_scenario(v) for k, v in TASK2_SCENARIOS.items()}).T
key2.loc["Reference (80 GBP/tCO2)"] = run_system_scenario({})
show_table(key2, 1)
save_table(key2, "instructor_key_task2_scenarios")

### 5.7 Stretch task: at what carbon price do coal and gas swap places?

Coal and gas have the same marginal cost when

$$35 + 0.90\,C = 60 + 0.37\,C \quad\Rightarrow\quad C = \frac{60 - 35}{0.90 - 0.37} \approx 47 \text{ GBP/tCO}_2$$

This is the **fuel-switching price**. The cell below checks it by solving the model for carbon prices from 0 to 150 GBP/tCO2 (about 20 seconds), and confirms that the model switches between 47 and 48 GBP/tCO2. **Figure 8** shows the step change in emissions at the switching price, while the peak price rises steadily, because gas sets the peak price throughout. The step is a result of this model, with only one coal unit and one gas unit. A real system has many plants with different efficiencies, so it responds more gradually.

In [ ]:
switch_price = (GENERATORS.loc["Gas", "srmc_excl_carbon_gbp_mwh"] - GENERATORS.loc["Coal", "srmc_excl_carbon_gbp_mwh"]) / (
    GENERATORS.loc["Coal", "emission_intensity_tco2_mwh"] - GENERATORS.loc["Gas", "emission_intensity_tco2_mwh"])
print(f"Analytical switching price: {switch_price:.1f} GBP/tCO2\n")

carbon_prices = list(range(0, 151, 10)) + [45, 46, 47, 48, 49]
sweep2 = pd.DataFrame([summarise_run(build_and_solve(c), c) for c in sorted(set(carbon_prices))])
sweep2 = sweep2.set_index("Carbon price (GBP/tCO2)")

# Verification: the numerical model agrees with the analytical switching price.
assert sweep2.loc[47, "Coal generation (MWh)"] > 0 and sweep2.loc[48, "Coal generation (MWh)"] < 1, \
    "Coal should leave the merit order between 47 and 48 GBP/tCO2."
print("Check passed: the model switches from coal to gas between 47 and 48 GBP/tCO2.\n")

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(sweep2.index, sweep2["Direct CO2 emissions (tCO2)"] / 1000, color=COLOURS["coal"], marker="o", linewidth=1.8,
         label="Daily direct CO2 emissions")
ax1.axvline(switch_price, color=COLOURS["gas"], linestyle=":", linewidth=1.2)
ax1.text(switch_price + 2, ax1.get_ylim()[1] * 0.95, f"Coal and gas swap\nat {switch_price:.0f} GBP/tCO2",
         fontsize=8, va="top", color=COLOURS["gas"])
ax1.set_xlabel("Carbon price (GBP/tCO2)")
ax1.set_ylabel("Daily direct CO2 emissions (thousand tCO2)")
ax2 = ax1.twinx()
ax2.plot(sweep2.index, sweep2["Peak-hour price (GBP/MWh)"], color=COLOURS["price"], linestyle="--", linewidth=1.5,
         label="Peak-hour price")
ax2.set_ylabel("Peak-hour price (GBP/MWh)")
ax2.grid(False)
ax2.spines["right"].set_visible(True)
handles = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax1.legend(handles, labels, loc="center right")
ax1.set_title("Figure 8. Emissions fall in a step; the price rises steadily", loc="left")
fig.tight_layout()
save_figure(fig, "fig08_carbon_price_sweep")
plt.show()

show_table(sweep2[["Peak-hour price (GBP/MWh)", "Price-setting unit at peak", "Direct CO2 emissions (tCO2)",
                   "Coal generation (MWh)", "Gas generation (MWh)"]], 1)
save_table(sweep2, "tab12_carbon_price_sweep")

**Discuss in your group (two minutes):**

1. The peak price rose even though gas set it in both runs. Why?
2. Consumer payments rose by more than the carbon cost. Where did the extra money go?
3. Emissions fell in a single step rather than gradually. What would make the response smoother in a real system?
4. The Carbon Price Support is due to be removed from April 2028. What would that do in this model?

## 6. Summary, limitations and downloads

### What you have done

- Loaded and checked a year of modelled hourly data, and built a battery model from a simple rule.
- Verified the model on a hand-calculated case and with energy balance checks.
- Showed that a battery raises the self-sufficiency rate by moving midday solar into the evening, that losses and power limits reduce the gain, and that returns diminish as the battery grows.
- Built a least-cost dispatch model in PyPSA and showed that a carbon price raises the clearing price through the marginal unit and cuts emissions when it moves high-carbon units behind lower-carbon units in the merit order.

Note that the checks in this notebook are **verification**: they show the code does what was intended. **Validation**, showing the model matches reality, would need measured data, such as smart meter readings and the output of a real PV system.

### Limitations

Every model is a simplification. Knowing what a model leaves out is as important as running it.

| Model | Assumption | Effect on the results |
|---|---|---|
| Task 1 | Smooth synthetic demand with hourly time steps | Hides short demand spikes, so self-consumption and self-sufficiency are **overstated** |
| Task 1 | Modelled PV output for one weather year (2019) | Measured systems usually produce less; other years differ by several per cent |
| Task 1 | Greedy battery rule, not an optimised schedule | Reasonable for a flat tariff, but **understates** the value of a battery on a time-of-use tariff |
| Task 1 | No battery degradation or standby losses | Slightly **overstates** the battery's benefit |
| Task 1 | Flat import and export prices from October 2026 | Savings change as tariffs change |
| Task 2 | One bus with no transmission limits | Ignores congestion and regional price differences |
| Task 2 | No start-up costs, minimum stable generation or ramp limits | Makes dispatch more flexible than in reality, and makes the switching response a sharp step |
| Task 2 | One illustrative day, perfect foresight, no storage or interconnectors | Results show the mechanism, not a forecast |
| Task 2 | Illustrative SRMCs and direct CO2 emission intensities | Change the numbers, not the shape of the result |

### Where to go next

- **Data Science for Energy System Modelling**, a free course by Fabian Neumann (TU Berlin), covering Python, pandas and PyPSA in much more depth: https://fneum.github.io/data-science-for-esm
- **PyPSA documentation and examples**: https://docs.pypsa.org
- **PyPSA-IMG**, an open-source industrial microgrid model built on PyPSA: https://github.com/ammarahmedwafad/pypsa-img

### Download everything

Run the cell below to collect all figures, tables and the input data into one zip file, `workshop_outputs.zip`.

- **JupyterLab (Anaconda):** the zip file is saved in the same folder as this notebook. Find it in the File Browser on the left, or in that folder on your computer.
- **Google Colab:** the download starts automatically. If it does not, open the Files panel on the left, right-click the zip file and choose **Download**.

In [ ]:
shutil.copy(DATA_DIR / "home.csv", Path("outputs") / "home.csv")
archive = shutil.make_archive("workshop_outputs", "zip", root_dir=".", base_dir="outputs")
print(f"Created {archive}")
for path in sorted(Path("outputs").rglob("*.*")):
    print(" ", path)

if "google.colab" in sys.modules:      # in Colab, download the zip straight away
    from google.colab import files
    files.download(archive)

## 7. References

1. Brown, T., Hörsch, J. and Schlachtberger, D. (2018) 'PyPSA: Python for Power System Analysis', *Journal of Open Research Software*, 6(1), p. 4. https://doi.org/10.5334/jors.188
2. Department for Energy Security and Net Zero (2025) 'Feed-in Tariff load factor analysis, 2024/25', *Energy Trends*, special article, 18 December 2025. https://www.gov.uk/government/publications/energy-trends-december-2025-special-feature-article-feed-in-tariff-load-factor-analysis-20242025
3. European Commission Joint Research Centre (2026) *Photovoltaic Geographical Information System (PVGIS)*, version 5.3. https://joint-research-centre.ec.europa.eu/photovoltaic-geographical-information-system-pvgis_en
4. Hofmann, F. (2023) 'Linopy: Linear optimization with n-dimensional labeled variables', *Journal of Open Source Software*, 8(84), p. 4823. https://doi.org/10.21105/joss.04823
5. Huangfu, Q. and Hall, J.A.J. (2018) 'Parallelizing the dual revised simplex method', *Mathematical Programming Computation*, 10(1), pp. 119 to 142. https://doi.org/10.1007/s12532-017-0130-5
6. IPCC (2006) *2006 IPCC Guidelines for National Greenhouse Gas Inventories, Volume 2: Energy*, chapter 2, Stationary combustion. Hayama: IGES.
7. IPCC (2014) 'Annex III: Technology-specific cost and performance parameters', in *Climate Change 2014: Mitigation of Climate Change. Contribution of Working Group III to the Fifth Assessment Report*. Cambridge: Cambridge University Press.
8. Luthander, R., Widén, J., Nilsson, D. and Palm, J. (2015) 'Photovoltaic self-consumption in buildings: a review', *Applied Energy*, 142, pp. 80 to 94. https://doi.org/10.1016/j.apenergy.2014.12.028
9. Ofgem (2026a) *Decision on revised Typical Domestic Consumption Values*. London: Ofgem. https://www.ofgem.gov.uk/sites/default/files/2026-06/Summary-of-changes-to-energy-price-cap-1-July-to-30-September-2026-revised-TDCV.pdf
10. Ofgem (2026b) *Energy price cap, 1 October to 31 December 2026*. London: Ofgem. https://www.ofgem.gov.uk/energy-price-cap
11. Pfeifroth, U. et al. (2024) *Surface Radiation Data Set, Heliosat (SARAH), edition 3*. EUMETSAT Satellite Application Facility on Climate Monitoring. https://doi.org/10.5676/EUM_SAF_CM/SARAH/V003
12. The pandas development team (2020) *pandas-dev/pandas: Pandas*. Zenodo. https://doi.org/10.5281/zenodo.3509134
13. UK Parliament (2026) *Carbon Price Support*, written ministerial statement HCWS1519, 16 April 2026.